# Cross-category causal patching + controls
Source = **'two cats'**. For a fixed seed we inject the early up0 self-attention of four donors and score the output for BOTH cats and dogs.

- `same-cat_cross-count` (five cats): positive control - count should rise, cats.
- `KEY_cross-cat_cross-count` (five dogs): **the crux** - if output = ~5 CATS, count transfers without category (abstract count/layout code); if dogs appear, category leaks (scene template).
- `ctrl_*_same-count` (two cats / two dogs): count should NOT move (isolates the donor-count effect from mere patch perturbation; and tests category leakage without a count change).

**Runtime:** GPU (~12 min).

In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/serinaqin/T2I-Count-Anomaly.git
    %cd T2I-Count-Anomaly
!pip install -q -r requirements.txt
!pip install -q pytest groundingdino-py

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd, os, yaml
import matplotlib.pyplot as plt
from src.prompts import build_prompt
from src.pipeline import (load_sdxl, generate, catalog_attention_sites,
                          select_probe_sites, generate_and_capture,
                          raw_reducer, generate_with_patch)
from src.detector import Detector
from src.scoring import count_from_detections
from src.config import load_config

In [ ]:
cfg = load_config('configs/exp_crosscat.yaml')
raw = yaml.safe_load(open('configs/exp_crosscat.yaml'))
psteps = raw['patch_steps']; block, attn = raw['patch_block'], raw['patch_attn']
src_cat, src_count, donors = raw['source_cat'], raw['source_count'], raw['donors']
cats = raw['objects']  # ['cat','dog'] - the two labels we always score
pipe = load_sdxl(); det = Detector()
sites = [s for s in select_probe_sites(catalog_attention_sites(pipe.unet))
         if block in s and s.endswith(attn)]
def counts_both(img):
    dets = det.detect(img, cats)
    return {a: count_from_detections(dets, a, cfg.score_threshold) for a in cats}
print('sites:', sites, '| labels scored:', cats)

In [ ]:
rows = []
for seed in cfg.seeds:
    sp = build_prompt(src_count, src_cat)          # 'two cats'
    b = generate(pipe, sp, seed, cfg.num_inference_steps)
    cb = counts_both(b)
    rows.append({'seed': seed, 'cond': 'baseline', 'donor_count': src_count,
                 'donor_cat': src_cat, **{f'out_{a}': cb[a] for a in cats}})
    for dcount, dcat, label in donors:
        dp = build_prompt(dcount, dcat)
        _, snaps = generate_and_capture(pipe, dp, seed, sites, psteps,
                                        cfg.num_inference_steps, reducer=raw_reducer)
        p = generate_with_patch(pipe, sp, seed, snaps, cfg.num_inference_steps)
        cp = counts_both(p)
        rows.append({'seed': seed, 'cond': label, 'donor_count': dcount,
                     'donor_cat': dcat, **{f'out_{a}': cp[a] for a in cats}})
    print('seed', seed, 'done')
df = pd.DataFrame(rows)
os.makedirs('results', exist_ok=True)
df.to_csv('results/exp_crosscat.csv', index=False)
df.groupby('cond')[[f'out_{a}' for a in cats]].mean().round(2)

In [ ]:
# Summary: for each condition, mean source-cat count and mean donor-cat (dog) count.
summ = df.groupby(['cond', 'donor_cat', 'donor_count'])[[f'out_{a}' for a in cats]].mean().round(2)
print(summ)
base = df[df.cond == 'baseline'][[f'out_{a}' for a in cats]].mean()
print('\nbaseline (two cats):', {a: round(base[f'out_{a}'], 2) for a in cats})

In [ ]:
# Bar: output cat-count vs dog-count per condition (the crux is cross-cat/cross-count).
m = df.groupby('cond')[[f'out_{a}' for a in cats]].mean()
order = ['baseline', 'ctrl_same-cat_same-count', 'same-cat_cross-count',
         'ctrl_cross-cat_same-count', 'KEY_cross-cat_cross-count']
m = m.reindex([o for o in order if o in m.index])
ax = m.plot(kind='bar', figsize=(10, 5))
ax.set_ylabel('mean output count'); ax.set_title('Cross-category patch: does COUNT transfer without CATEGORY?')
ax.legend(title='detected as'); plt.xticks(rotation=20, ha='right')
plt.tight_layout(); plt.savefig('results/exp_crosscat_bars.png', dpi=100, bbox_inches='tight'); plt.show()

In [ ]:
# Eyeball: baseline vs each donor for one seed. Are the added animals cats or dogs?
seed0 = cfg.seeds[0]; sp = build_prompt(src_count, src_cat)
panels = [('baseline', generate(pipe, sp, seed0, cfg.num_inference_steps))]
for dcount, dcat, label in donors:
    _, snaps = generate_and_capture(pipe, build_prompt(dcount, dcat), seed0, sites,
                                    psteps, cfg.num_inference_steps, reducer=raw_reducer)
    panels.append((f'{dcount} {dcat}', generate_with_patch(pipe, sp, seed0, snaps, cfg.num_inference_steps)))
fig, axes = plt.subplots(1, len(panels), figsize=(3.0 * len(panels), 3.3))
for ax, (name, im) in zip(axes, panels):
    cc = counts_both(im)
    ax.imshow(im); ax.axis('off')
    ax.set_title(f'{name}\ncat={cc[cats[0]]} dog={cc[cats[1]]}', fontsize=8)
plt.tight_layout(); plt.savefig('results/exp_crosscat_eyeball.png', dpi=90, bbox_inches='tight'); plt.show()

## How to read this
Focus on **KEY_cross-cat_cross-count** (donor = five dogs, source = two cats):
- **Output ~5 CATS, ~0 dogs** -> the count/layout transfers but the category does NOT -> an abstract, category-invariant **count/layout code**. Strong result; 'count code' is earned.
- **Output shows DOGS (or a cat/dog mix scaling with the donor)** -> the donor transplanted a *scene/instance layout* tied to its content -> reframe from 'count code' to **object-layout code** (count is emergent). Also a clean, publishable conclusion - just a different one.

**Controls:** `ctrl_*_same-count` (two cats / two dogs donor) should leave the cat-count near the baseline (~2): if it does, the count shift in the cross-count conditions is driven by the donor's COUNT, not by mere patch perturbation. If the 'two dogs' control makes dogs appear without changing the count, category leaks spatially even without a count change (informative for the layout-vs-count question).